In [91]:
from collections.abc import Sequence
import re

from langchain_core.documents import Document

# Markdown 标题常用分隔符：更长者优先，避免 `##` 被 `#` 抢先匹配
DEFAULT_MD_SEPARATORS = ("\n## ", "\n# ")


class SeparatorsSplitter:
    def __init__(
        self,
        separators: Sequence[str] = DEFAULT_MD_SEPARATORS,
        keep_separator: bool = True,
        max_size: int | None = None,
    ):
        self.separators = separators
        self.keep_separator = keep_separator
        self.max_size = max_size

    def _pack_by_size(self, text: str, max_size: int) -> list[str]:
        """超长块二次切分：优先按空行拼段，单段仍超长再硬切。"""
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        if not paragraphs:
            return [text.strip()] if text.strip() else []

        chunks: list[str] = []
        buf = ""
        for p in paragraphs:
            candidate = f"{buf}\n\n{p}".strip() if buf else p
            if len(candidate) <= max_size:
                buf = candidate
                continue
            if buf:
                chunks.append(buf)
            if len(p) <= max_size:
                buf = p
            else:
                for i in range(0, len(p), max_size):
                    piece = p[i : i + max_size].strip()
                    if piece:
                        chunks.append(piece)
                buf = ""
        if buf:
            chunks.append(buf)
        return chunks

    def split_text_by_separators(self, text: str) -> list[str]:
        """先按 separators 严格切开；若设了 max_size，超长块再按长度二次切分。

        - 只在给定 separators 处做「语义切」
        - 更长的 separator 优先匹配（如 ``\\n## `` 优先于 ``\\n# ``）
        - ``keep_separator=True`` 时，分隔符保留在下一块开头
        - ``max_size`` 为 None：不按长度再切；否则超长节走段落打包 / 硬切
        """
        text = text or ""
        if not text.strip():
            return []

        seps = sorted((s for s in self.separators if s), key=len, reverse=True)
        if not seps:
            sections = [text.strip()]
        else:
            pattern = "|".join(re.escape(s) for s in seps)
            if self.keep_separator:
                parts = re.split(f"({pattern})", text)
                sections = []
                prefix = parts[0]
                if prefix.strip():
                    sections.append(prefix.strip())
                i = 1
                while i < len(parts):
                    sep = parts[i]
                    body = parts[i + 1] if i + 1 < len(parts) else ""
                    head = sep[1:] if sep.startswith("\n") else sep
                    chunk = f"{head}{body}".strip()
                    if chunk:
                        sections.append(chunk)
                    i += 2
            else:
                sections = [p.strip() for p in re.split(pattern, text) if p.strip()]

        if self.max_size is None:
            return sections

        chunks: list[str] = []
        for section in sections:
            if len(section) <= self.max_size:
                chunks.append(section)
            else:
                chunks.extend(self._pack_by_size(section, self.max_size))
        return chunks

    def split_documents(self, documents: list[Document]) -> list[Document]:
        out: list[Document] = []
        for doc in documents:
            for chunk in self.split_text_by_separators(doc.page_content):
                out.append(Document(page_content=chunk, metadata=dict(doc.metadata)))
        return out




In [95]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, BSHTMLLoader, CSVLoader
# from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter, CharacterTextSplitter
# from pprint import pprint

def get_loader(file_path: str):
  if (file_path.endswith('.md')):
    return TextLoader(file_path=file_path)
  elif (file_path.endswith('.html')):
    return BSHTMLLoader(file_path=file_path)
  elif (file_path.endswith('.csv')):
    return CSVLoader(file_path=file_path)
#   else:
#     return DirectoryLoader(file_path=file_path,  glob=['*.md', '*.html', '*.csv'], loader_cls=get_loader)


def load_documents(path: str):
  loader = DirectoryLoader(
    path=path,
    glob=['*.md', '*.html', '*.csv'],
    loader_cls=get_loader,
    recursive=True
  )
  return loader.load()

documents = load_documents('/Users/linqibin/Documents/coding/rag-knowledge-assistant/data/corpus/internal')

# header_splitter = MarkdownHeaderTextSplitter(
#     headers_to_split_on=[
#         ("#", "h1"),
#         ("##", "h2"),
#     ],
#     strip_headers=False
# )

# md_docs = []

# for doc in documents:
#     splits = header_splitter.split_text(doc.page_content)
#     print(splits)
#     print('-' * 100)
#     # md_docs.extend(splits)

# docs = RecursiveCharacterTextSplitter(
#   separators=['\n### ', '\n## ', '\n# '],
#   chunk_size=200,
#   chunk_overlap=0,
# ).split_documents(documents)


chunks = SeparatorsSplitter(
  separators=['\n### ', '\n## ', '\n# '],
  keep_separator=True,
  # max_size=200,  # None=只按标题切；设了则超长节再按段落/硬切
).split_documents(documents)


In [96]:
from colorama import Fore

for chunk in chunks:
    print(Fore.BLUE + "=" * 100 + Fore.RESET + '\n')
    print(Fore.GREEN + str(chunk.page_content) + Fore.RESET + '\n')
print(Fore.BLUE + "=" * 100 + Fore.RESET + '\n')


# 星云科技 · 内部知识库语料（虚构）

本目录为 **公司内部文档风** 练习语料，内容虚构，仅供 RAG 学习。

| 子目录 | 格式 | 状态 |
|--------|------|------|
| `markdown/` | 制度、FAQ、SOP（中文） | 已接入 |
| `html/` | 模拟内部 Wiki 页面 | 已接入 |
| `csv/` | 通讯录 / 系统清单 | 已接入 |
| `pdf/` | 制度 PDF（放入文件后再加解析） | 预留 |

默认 ingest 根目录：`data/corpus/internal`


# IT 与账号管理规范

**文档编号**：IT-ACC-004  
**生效日期**：2025-01-20  
**责任部门**：信息技术中心


## 1. 入职账号

新员工入职当天由 IT 开通：

- 企业邮箱：`姓名拼音@xingyun-example.com`
- 飞书 / 即时通讯
- 假勤系统、费控系统、Git 只读权限（研发岗另开通写权限）
- VPN 账号（默认关闭，需申请）

初始密码通过短信发送，**首次登录必须修改**；禁止转发短信给他人。


## 2. 权限申请

权限变更走 **ITSM 工单**：https://itsm.xingyun-example.com

| 权限类型 | 审批人 | 时效 |
|---------|--------|------|
| 业务系统只读 | 直属上级 | 1 个工作日 |
| 生产环境只读 | 上级 + 业务负责人 | 2 个工作日 |
| 生产环境写 / 发布 | 上级 + 安全接口人 | 3 个工作日 |
| 管理员角色 | CTO 或指派负责人 | 一事一议 |


## 3. VPN 与远程访问

- 仅限处理生产故障、紧急发布或合规审计
- 申请单需写明事由与预计使用天数；到期自动关闭
- 禁止在非公司发放的个人电脑上保存客户数据导出文件


## 4. 离职与调岗

- 离职最后工作日 18:00 前，IT 禁用全部账号
- 调岗：原部门权限在 3 个工作日内回收，新部门权限重新申请
- 共用账号（如值班号）须在 ITSM 登记责任人，禁止个人长期占用


## 5. 密码与 MFA

- 密码长度 ≥ 12，每 90 天轮